我们在原有的 02_Stream real time monitor 里面往下递进，利用F.window强行切出1分钟滚动窗口，在流中实时榨取GMV交易总额和Order_Count总单量

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

print("=== [METRICS MONITOR] 启动分钟级核心 GMV 与战报指标提炼... ==")

In [0]:
# 接通Volume活水网络

# 1. 强行固化 Schema 防火墙
order_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", StringType(), True),
    StructField("status", StringType(), True)
])

df_stream = spark.readStream \
    .schema(order_schema) \
    .parquet("/Volumes/dbacademy/streaming/vol_yuto_stream_source/")

In [0]:
# 前置防腐拦截：只抓取高危或成功大单，让内存前置瘦身

# df_alerts = df_stream.filter(
#     (F.col("amount") > 10000.0) | (F.col("status") == "FAILED")
# )

In [0]:
# 🚀 核心重构：升级为1分钟滚动窗口聚合（Tumbling Window）
# 统计每分钟内，各个 client_id 产生的异常订单量与高危GMV波动

# df_minute_metrics = df_alerts \
#     .withColumn("event_timestamp", F.to_timestamp("event_time")) \
#     .withWatermark("event_timestamp", "5 minutes") \
#     .groupBy(
#         F.window(F.col("event_timestamp"), "1 minute"), #  1分钟一刀，无重叠滚动
#         F.col("client_id")
#     ).agg(
#         F.sum("amount").alias("minute_gmv"),            #  统计分钟级实时 GMV
#         F.count("order_id").alias("minute_order_count")  #  统计分钟级实时订单量
#     ).select(
#         F.col("window.start").alias("window_start"),     # 展开窗口时空开始时间
#         F.col("window.end").alias("window_end"),         # 展开窗口时空结束时间
#         F.col("client_id"),
#         F.col("minute_gmv"),
#         F.col("minute_order_count")
#     )

df_minute_metrics = df_stream \
    .withColumn("event_timestamp", F.to_timestamp("event_time")) \
    .withWatermark("event_timestamp", "5 minutes") \
    .groupBy(
        F.window(F.col("event_timestamp"), "1 minute"), #  1分钟一刀，无重叠滚动
        F.col("client_id")
    ).agg(
        F.sum("amount").alias("minute_gmv"),            #  统计分钟级实时 GMV
        F.count("order_id").alias("minute_order_count")  #  统计分钟级实时订单量
    ).select(
        F.col("window.start").alias("window_start"),     # 展开窗口时空开始时间
        F.col("window.end").alias("window_end"),         # 展开窗口时空结束时间
        F.col("client_id"),
        F.col("minute_gmv"),
        F.col("minute_order_count")
    )

In [0]:
checkpoint_gold_path = "/Volumes/dbacademy/streaming/vol_yuto_stream_source/_checkpoints/gold_metrics_monitor/"

gold_table_path = "/Volumes/dbacademy/streaming/vol_yuto_stream_source/gold_realtime_minute_dashboard"



# 强制递归删除旧的、冲突的检查点账本，让新版分钟级滚动窗口引擎能够轻装上阵
# dbutils.fs.rm(checkpoint_minute_path, recurse=True)

# 核心重构：发动持久化流式引擎，轰入 Delta Gold 物理表
# 为了支持 complete 模式写入文件系统，我们必须使用 path 或者注册成 Table
query_gold = df_minute_metrics.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", checkpoint_gold_path) \
    .trigger(availableNow=True) \
    .start(gold_table_path) # 👈 直接落盘到 Volume 的物理路径下！

query_gold.awaitTermination()
print("🎉 [实时黄金表固化成功] 分钟级全量业务指标与风控隔离资产已铁血落盘！")

In [0]:
# print(" === 实时分钟级高危GMV战报核心大盘 ===\n")

# # 拉取大盘视图并按照时间线正序排开

# df_dashboard = spark.table("yuto_realtime_minute_dashboard")

# df_dashboard.sort("window_start", "client_id").show(truncate=False)

# # 计算历史累计被拦截的总 GMV
# total_intercepted_gmv = df_dashboard.agg(F.sum("minute_gmv")).collect()[0][0] or 0.0
# print(f"【大盘资产总账】: 当前实时雷达已累计监控并锁死的高危/失败 GMV 资产总额: {total_intercepted_gmv:.2f} 元。")
